[SamplingBallsBinomialBootstrapping.ipynb](https://colab.research.google.com/drive/1ogFWga0l7dPigeak_JmSTCARVJFeMUco?authuser=1#scrollTo=efc5dad3)
[Drawing From Balls - Spreadsheet Result](https://docs.google.com/spreadsheets/d/1itsWZRKIU8xtnzKUfgyDgSe07tu6OL4nN1ab3s4OkPo/edit?gid=0#gid=0


In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from typing_extensions import TypeAlias
import numpy as np
import pandas as pd

from phonegativerelativeprobabilities.sampling import (
    bag_to_observation_string,
    generate_bag,
    observation_string_to_bag,
    sample_from_bag,
    repeat_sample_from_bag
)


# ==================================================================================================================================================================================================================================================================================== #
# BEGIN BODY                                                                                                                                                                                                                                                                           #
# ==================================================================================================================================================================================================================================================================================== #
n_total: int = 25

# Generate a random bag
ball_color_prob_dict = {'orange': 3.0/float(n_total), 'white': (n_total - 3.0)/ float(n_total)}
generated_bag = generate_bag(n_total=n_total, ball_color_prob_dict=ball_color_prob_dict)
display(bag_to_observation_string(generated_bag))
samples = sample_from_bag(bag_of_balls=generated_bag)
display(bag_to_observation_string(samples))

# Generate a fixed bag
ball_color_counts_dict = {'orange': 3, 'white': (n_total - 3)}
generated_bag = generate_bag(n_total=n_total, ball_color_counts_dict=ball_color_counts_dict)
display(bag_to_observation_string(generated_bag))


Automatic pdb calling has been turned OFF


'wwoowwowwwwowwwwwwwwwwwww'

'wwwwoowwowwwwwwwwwwwwwwow'

'wwwwowwwwwwwowwwwwwowwwww'

In [4]:
n_repeats: int = 1000000
samples, color_counts_df = repeat_sample_from_bag(generated_bag, n_repeats=1000000)  # list[list[str]], shape ~ (1000, n_samples)

# samples
color_counts_df

,orange,white
0,3,22
1,3,22
2,4,21
3,3,22
4,3,22
...,...,...
999995,3,22
999996,3,22
999997,3,22
999998,3,22


In [ ]:
matching_condition_df = color_counts_df[color_counts_df['orange'] == 4]
conditional_probability: float = float(len(matching_condition_df)) / float(n_repeats)
conditional_probability # 0.173478

0.173478

In [8]:
# cc_df: pd.DataFrame = pd.DataFrame.from_records(color_counts, columns=['n_orange', 'n_white'])
cc_df = pd.DataFrame(color_counts)
cc_df

,orange,white
0,4,21
1,4,21
2,4,21
3,4,21
4,4,21
...,...,...
995,5,20
996,5,20
997,4,21
998,4,21


In [ ]:
cc_df.mean()


orange     4.177
white     20.823
dtype: float64

In [ ]:
obs_seq_string_example_original_paper: str = 'owowwowwwwwwwwwwwwwwwowww'
example_bag_from_paper = observation_string_to_bag(obs_seq_string_example_original_paper)
# display(example_bag_from_paper)
print(pd.Series(example_bag_from_paper).value_counts().to_dict())

roundtrip_obs_seq = bag_to_observation_string(example_bag_from_paper)
assert roundtrip_obs_seq == obs_seq_string_example_original_paper
print(roundtrip_obs_seq)


In [ ]:
measurement_noise_probabilities: pd.DataFrame = pd.DataFrame([[0.99, 0.01], [0.01, 0.99]], columns=['real_white', 'real_orange'], index=['obs_white', 'obs_orange'])
measurement_noise_probabilities

In [ ]:
def observe_ball(real_color: str, noise_df: pd.DataFrame):
    """
    Simulates observing a ball with measurement noise.
    """
    # Map the real color name to the column name in the noise dataframe
    column_name = f'real_{real_color}'

    # Get the possible observations and their probabilities
    observations = noise_df.index.tolist()
    probabilities = noise_df[column_name].values

    # Randomly choose an observation
    observation = np.random.choice(observations, p=probabilities)

    # Strip the 'obs_' prefix to return just the color
    return observation.replace('obs_', '')

# Demonstrate observing the entire bag
observed_bag = [observe_ball(ball, measurement_noise_probabilities) for ball in generated_bag]

print(f"Real bag (first 10): {generated_bag[:10]}")
print(f"Observed bag (first 10): {observed_bag[:10]}")

# Compare counts
print("\nReal Counts:", pd.Series(generated_bag).value_counts().to_dict())
print("Observed Counts:", pd.Series(observed_bag).value_counts().to_dict())

In [ ]:
Implement an observation event, which consumes a ball in a bag and returns the color of the ball with a certain correctness probability, simulating measurement error.
The user defines a table `measurement_noise_probabilities: pd.DataFrame = pd.DataFrame([[0.99, 0.01], [0.01, 0.99]], columns=['real_white', 'real_orange'], index=['obs_white', 'obs_orange'])` which gives the probabilities for observing a color when the drawn color was really a color.